<a href="https://colab.research.google.com/github/esthy13/cil-intrusion-detection/blob/main/notebooks/1_er_strategy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Experience Replay

In [1]:
from google.colab import userdata
token = userdata.get('GIT_TOKEN')
username = userdata.get('USERNAME')
email = userdata.get('EMAIL')
!git config --global user.email {email}
!git config --global user.name {username}
repo = "cil-intrusion-detection"

In [2]:
!git clone https://{token}@github.com/{username}/{repo}
%cd {repo}
!git pull origin main

fatal: destination path 'cil-intrusion-detection' already exists and is not an empty directory.
/content/cil-intrusion-detection
From https://github.com/esthy13/cil-intrusion-detection
 * branch            main       -> FETCH_HEAD
Already up to date.


In [3]:
import sys
import os
import torch
import torch.nn as nn
import numpy as np
import random

from torch.utils.data import DataLoader, Subset

# Project files
from src.dataset import UNSWDataset
from src.model import CILModel
from src.task_builder import build_task, build_scenario, UpToNormalizer
from src.metrics import accuracy, macro_f1, compute_cm, save_confusion_matrix, average_accuracy
from src.utils import (
    print_task_results,
    print_scenario,
    print_strategy,
    save_training_results
)

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)

Using device: cuda


## Loading Datasets

TODO edit code so that you can simply call the main method to run everything. check the path used to save files in the results folders both for Jean's part and for Esther's part!

In [4]:
# !unzip data/processed/2017.zip -d data/processed/
# !unzip data/processed/2015.zip -d data/processed/

In [5]:
dataset_2015 = UNSWDataset.from_root_dir("data/processed/2015", "attack_cat", "Normal")
dataset_2017 = UNSWDataset.from_root_dir("data/processed/2017", "Label", "benign")

## 2015 Training and evaluate

In [8]:
from src.er import train_and_evaluate_ER

attack_pattern = [2, 2, 2, 2]

avg_acc, forgetting = train_and_evaluate_ER(
    # scenario_id="ER-[2,2,2,2]",
    trainset=dataset_2015,
    feature_dim=128,
    device=device,
    memory_size=2000,
    attack_pattern=attack_pattern,
    epochs=5
)


Strategy ExperienceReplay ========

=== Scenario - [2, 2, 2, 2] ===

   --- Task 1 ---

    New attacks: ['Normal', 'Exploits', 'Fuzzers']
    Seen so far: ['Normal', 'Exploits', 'Fuzzers']

    accuracy: 0.75
    attack_accuracy: 0.43
    macro-f1: 0.56


   --- Task 2 ---

    New attacks: ['Normal', 'Exploits', 'Fuzzers', 'Generic', 'Reconnaissance']
    Seen so far: ['Normal', 'Exploits', 'Fuzzers', 'Generic', 'Reconnaissance']

    accuracy: 0.67
    attack_accuracy: 0.48
    macro-f1: 0.60


   --- Task 3 ---

    New attacks: ['Normal', 'Exploits', 'Fuzzers', 'Generic', 'Reconnaissance', 'DoS', 'Analysis']
    Seen so far: ['Normal', 'Exploits', 'Fuzzers', 'Generic', 'Reconnaissance', 'DoS', 'Analysis']

    accuracy: 0.53
    attack_accuracy: 0.62
    macro-f1: 0.69


   --- Task 4 ---

    New attacks: ['Normal', 'Exploits', 'Fuzzers', 'Generic', 'Reconnaissance', 'DoS', 'Analysis', 'Shellcode', 'Backdoor']
    Seen so far: ['Normal', 'Exploits', 'Fuzzers', 'Generic', 'Reconn

In [10]:
from src.er import train_and_evaluate_ER

attack_pattern = [2, 2, 2, 1]

avg_acc, forgetting = train_and_evaluate_ER(
    # scenario_id="ER-[2,2,2,2]",
    trainset=dataset_2017,
    feature_dim=128,
    device=device,
    memory_size=2000,
    attack_pattern=attack_pattern,
    epochs=5
)


Strategy ExperienceReplay ========

=== Scenario - [2, 2, 2, 1] ===

   --- Task 1 ---

    New attacks: ['benign', 'dos', 'portscan']
    Seen so far: ['benign', 'dos', 'portscan']

    accuracy: 0.89
    attack_accuracy: 0.44
    macro-f1: 0.57


   --- Task 2 ---

    New attacks: ['benign', 'dos', 'portscan', 'ddos', 'ftp-patator']
    Seen so far: ['benign', 'dos', 'portscan', 'ddos', 'ftp-patator']

    accuracy: 0.70
    attack_accuracy: 0.53
    macro-f1: 0.61


   --- Task 3 ---

    New attacks: ['benign', 'dos', 'portscan', 'ddos', 'ftp-patator', 'ssh-patator', 'web-attack']
    Seen so far: ['benign', 'dos', 'portscan', 'ddos', 'ftp-patator', 'ssh-patator', 'web-attack']

    accuracy: 0.69
    attack_accuracy: 0.51
    macro-f1: 0.65


   --- Task 4 ---

    New attacks: ['benign', 'dos', 'portscan', 'ddos', 'ftp-patator', 'ssh-patator', 'web-attack', 'bot']
    Seen so far: ['benign', 'dos', 'portscan', 'ddos', 'ftp-patator', 'ssh-patator', 'web-attack', 'bot']

    accu

In [11]:
commit_message = "first batch of ER code reorganization works with both datasets" # @param {type:"string"}
!git remote set-url origin https://{token}@github.com/esthy13/cil-intrusion-detection.git
!git add .
!git status
!git commit -m "$commit_message"
!git push origin main

On branch main
Your branch is up to date with 'origin/main'.

Changes to be committed:
  (use "git restore --staged <file>..." to unstage)
	modified:   results/confusion_matrices/task_1_eval_on_task_1.png
	modified:   results/confusion_matrices/task_2_eval_on_task_1.png
	modified:   results/confusion_matrices/task_2_eval_on_task_2.png
	modified:   results/confusion_matrices/task_3_eval_on_task_1.png
	modified:   results/confusion_matrices/task_3_eval_on_task_2.png
	modified:   results/confusion_matrices/task_3_eval_on_task_3.png
	modified:   results/confusion_matrices/task_4_eval_on_task_1.png
	modified:   results/confusion_matrices/task_4_eval_on_task_2.png
	modified:   results/confusion_matrices/task_4_eval_on_task_3.png
	modified:   results/confusion_matrices/task_4_eval_on_task_4.png

[main 9eaa184] first batch of ER code reorganization works with both datasets
 10 files changed, 0 insertions(+), 0 deletions(-)
 rewrite results/confusion_matrices/task_1_eval_on_task_1.png (98%)
 re